# Laboratory Activity 1: Data Cleaning Pipeline
## Water Potability and Chemical Safety Assessment
**Domain:** Aquatic Chemistry & Public Health (UN Sustainable Development Goal 6)  
**Dataset:** `water_potability.csv`  

---

### Project Overview and Problem Statement
Clean drinking water is fundamental to human health and ecological sustainability. In aquatic quality monitoring, water safety classification relies on physicochemical assays (such as pH, Sulfates, Chloramines, and Trihalomethanes). However, environmental monitoring records frequently suffer from:
1. **Selective Laboratory Test Omission** (missing assay values for expensive or specialized tests).
2. **Sensor Hardware and Calibration Malfunctions** (physically out-of-scale pH probes).
3. **Extreme Geochemical Concentration Spikes** (anomalous mineral readings vs. high-salinity natural aquifers).

### Core Investigation Question
> *How can we accurately identify and clean sensor probe anomalies, laboratory test omissions, and mineral concentration outliers across aquatic physicochemical parameters to ensure that water safety classification strictly adheres to World Health Organization (WHO) safety standards?*

---
### Dataset Variable Codebook

| Variable | Raw Data Type | Unit / Scale | WHO Reference Standard / Environmental Significance |
| :--- | :--- | :--- | :--- |
| `ph` | `float64` | Scale (0 - 14) | Acid-base balance of water (WHO safe range: 6.5 - 8.5) |
| `Hardness` | `float64` | mg/L | Calcium and Magnesium mineral precipitation capacity |
| `Solids` | `float64` | ppm | Total Dissolved Solids (TDS); mineralization level |
| `Chloramines` | `float64` | ppm | Disinfectant agent used in municipal treatment (WHO limit: $\le$ 4 ppm) |
| `Sulfate` | `float64` | mg/L | Naturally occurring minerals from geological runoff |
| `Conductivity` | `float64` | $\mu$S/cm | Indicator of dissolved ionic substances and mineralization |
| `Organic_carbon` | `float64` | ppm | Total organic carbon content from decaying natural matter |
| `Trihalomethanes` | `float64` | $\mu$g/L | Chlorine disinfection byproducts; potential carcinogens (WHO limit: $\le$ 80 $\mu$g/L) |
| `Turbidity` | `float64` | NTU | Suspended colloidal matter affecting water clarity (WHO limit: $\le$ 5 NTU) |
| `Potability` | `int64` | Binary (0 / 1) | Target safety label (0 = Non-Potable / Unsafe, 1 = Potable / Safe) |

---
## 1. Environment Setup and Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import os

# Visual styling configuration for analytical charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Environment initialized successfully.")

---
## 2. Raw Data Ingestion and Initial Profiling

In [ ]:
# Load raw dataset
raw_data_path = "water_potability.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

### Initial Data Profiling and Quality Audit

In [ ]:
# Summary information and data types
df_raw.info()

In [ ]:
# Descriptive statistical summary
df_raw.describe().T

In [ ]:
# Audit missing values across all features
missing_summary = pd.DataFrame({
    'Missing Count': df_raw.isna().sum(),
    'Missing Percentage (%)': (df_raw.isna().sum() / len(df_raw) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
missing_summary

In [ ]:
# Missing value matrix visualization
plt.figure(figsize=(10, 4))
msno.matrix(df_raw, figsize=(10, 4), color=(0.2, 0.4, 0.6), fontsize=10)
plt.title('Missing Value Matrix in Raw Dataset', fontsize=13, fontweight='bold', pad=15)
plt.show()

In [ ]:
# Missing value percentage chart
plt.figure(figsize=(8, 4))
ax = sns.barplot(
    x=missing_summary.index,
    y=missing_summary['Missing Percentage (%)'],
    palette='crest'
)
plt.title('Missing Value Percentage by Parameter in Raw Dataset', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Missing Percentage (%)')
plt.xlabel('Chemical Parameter')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
plt.ylim(0, 30)
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning Step 1: Enforce Physicochemical pH Boundaries (0 to 14)

### Domain Rationale:
* **The Science:** The pH scale measures the negative decimal logarithm of hydrogen ion activity ($-\log_{10}[H^+]$) in aqueous solutions. In natural and treated aquatic systems, pH strictly exists between **0 and 14**.
* **The Anomaly:** Readings below 0 or above 14 represent sensor electrode degradation, voltage drift, or calibration failure. They cannot represent real chemical observations and must be sanitized to `NaN` prior to imputation.

In [ ]:
df = df_raw.copy()

# Identify out-of-bound pH readings
invalid_ph_mask = (df["ph"] < 0) | (df["ph"] > 14)
print(f"Out-of-scale pH records detected: {invalid_ph_mask.sum()}")

# Set invalid readings to NaN
df.loc[invalid_ph_mask, "ph"] = np.nan
print("pH physical boundary constraint (0 <= pH <= 14) enforced.")

---
## 4. Data Cleaning Step 2: Class-Conditional Median Imputation for Chemical Assays

### Domain Rationale:
1. **Why Not `dropna()`?**
   * Over 30% of the samples have at least one missing chemical assay (`Sulfate` is missing ~23.8%, `ph` ~15.0%, `Trihalomethanes` ~4.95%).
   * Dropping rows would discard over **1,200 valid observations**, drastically reducing sample size and introducing severe **selection bias** (since missingness is due to selective testing of water sources).
2. **Why the Median instead of the Mean?**
   * Geochemical and microbial concentrations exhibit skewed, heavy-tailed distributions due to localized mineral runoff. The median is robust against extreme outliers.
3. **Why Class-Conditional (grouped by `Potability`)?**
   * Potable and non-potable water exhibit distinct biochemical and mineral baselines. Imputing separately per class avoids cross-class contamination and preserves the statistical separation between safe and unsafe water.

In [ ]:
# Distribution before imputation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
impute_features = ["ph", "Sulfate", "Trihalomethanes"]

for i, col in enumerate(impute_features):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Raw Distribution: {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Apply Class-Conditional Median Imputation
for col in impute_features:
    median_potable = df[df['Potability'] == 1][col].median()
    median_non_potable = df[df['Potability'] == 0][col].median()
    print(f"[{col}] Imputation Median -> Non-Potable (0): {median_non_potable:.3f} | Potable (1): {median_potable:.3f}")
    
    df[col] = df.groupby("Potability")[col].transform(
        lambda group: group.fillna(group.median())
    )

# Fallback: fill any residual global nulls if any group was completely empty
for col in df.columns:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"\nRemaining Missing Values in Dataset: {df.isna().sum().sum()}")

In [ ]:
# Visualizing distributions after class-conditional imputation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(impute_features):
    sns.kdeplot(data=df, x=col, hue='Potability', common_norm=False, ax=axes[i], palette=['#e74c3c', '#2ecc71'])
    axes[i].set_title(f'Post-Imputation Distribution: {col}')
plt.tight_layout()
plt.show()

---
## 5. Data Cleaning Step 3: Domain-Informed Outlier Treatment (Total Dissolved Solids)

### Domain Rationale:
* **The Science:** Total Dissolved Solids (`Solids`) measure the aggregate mineral and inorganic salt content in water.
* **Geological Variance vs. Measurement Artifacts:**
  * Freshwater usually ranges from 100 to 1,000 ppm TDS.
  * Mineral-rich groundwater aquifers and coastal brackish estuaries naturally reach 10,000 to 30,000 ppm TDS.
  * A standard statistical rule of **$1.5 \times \text{IQR}$** would mistakenly discard genuine, high-mineral natural water sources.
  * Using a conservative threshold of **$3.0 \times \text{IQR}$** preserves real natural geochemical variance while pruning physically unfeasible sensor telemetry spikes (> 50,000 ppm, approaching oceanic brine).

In [ ]:
# Calculate 3.0 * IQR upper threshold for Solids
q1 = df["Solids"].quantile(0.25)
q3 = df["Solids"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 3.0 * iqr

extreme_solids = df[df["Solids"] > upper_limit]
print(f"Q1: {q1:.2f} ppm | Q3: {q3:.2f} ppm | IQR: {iqr:.2f} ppm")
print(f"3.0 * IQR Upper Threshold: {upper_limit:.2f} ppm")
print(f"Extreme sensor spikes detected: {len(extreme_solids)} records")

In [ ]:
# Visualize Solids distribution and extreme cutoff
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['Solids'], color='lightblue')
plt.axvline(upper_limit, color='red', linestyle='--', label=f'3.0*IQR Threshold ({upper_limit:.1f} ppm)')
plt.title('Total Dissolved Solids (ppm) Outlier Detection', fontsize=13, fontweight='bold')
plt.xlabel('Solids (ppm)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Filter out extreme sensor spikes
df_cleaned = df[df["Solids"] <= upper_limit].reset_index(drop=True)
print(f"Rows before outlier pruning: {len(df)}")
print(f"Rows after outlier pruning:  {len(df_cleaned)}")

---
## 6. Data Cleaning Step 4: Data Type Integrity and Precision Normalization

### Actions:
1. Ensure the target column `Potability` is explicitly cast to `int64`.
2. Standardize continuous physicochemical readings by rounding to **3 decimal places** to eliminate floating-point machine precision artifacts.
3. Validate that zero missing values remain across the entire dataframe.

In [ ]:
# Enforce integer typing on target
df_cleaned["Potability"] = df_cleaned["Potability"].astype(int)

# Round continuous numerical metrics to 3 decimal places
continuous_cols = [c for c in df_cleaned.columns if c != "Potability"]
for col in continuous_cols:
    df_cleaned[col] = df_cleaned[col].round(3)

# Verification Assertions
assert df_cleaned.isna().sum().sum() == 0, "Error: Missing values remain!"
assert df_cleaned['Potability'].isin([0, 1]).all(), "Error: Potability has non-binary values!"
print("Verification passed: 0 missing values, valid data types, consistent precision.")
df_cleaned.info()

---
## 7. Data Cleaning Step 5: Export Cleaned Dataset

In [ ]:
output_csv_path = "water_potability_cleaned.csv"
df_cleaned.to_csv(output_csv_path, index=False)
print(f"Cleaned dataset successfully exported to: {output_csv_path}")

---
## 8. Before vs. After Data Cleaning Evaluation

In [ ]:
summary_comparison = pd.DataFrame({
    "Quality Metric / Check": [
        "Total Rows",
        "Total Missing Values",
        "Missing Sulfate Assays",
        "Missing pH Readings",
        "Missing Trihalomethanes",
        "Max Solids (ppm)",
        "Potable Samples (%)",
        "Data Types & Integrity"
    ],
    "Raw Dataset (water_potability.csv)": [
        f"{len(df_raw):,}",
        f"{df_raw.isna().sum().sum():,}",
        f"{df_raw['Sulfate'].isna().sum()} ({df_raw['Sulfate'].isna().mean()*100:.2f}%)",
        f"{df_raw['ph'].isna().sum()} ({df_raw['ph'].isna().mean()*100:.2f}%)",
        f"{df_raw['Trihalomethanes'].isna().sum()} ({df_raw['Trihalomethanes'].isna().mean()*100:.2f}%)",
        f"{df_raw['Solids'].max():.1f} ppm",
        f"{df_raw['Potability'].sum()} ({df_raw['Potability'].mean()*100:.2f}%)",
        "High Null Rate / Out-of-scale values"
    ],
    "Cleaned Dataset (water_potability_cleaned.csv)": [
        f"{len(df_cleaned):,}",
        f"{df_cleaned.isna().sum().sum():,}",
        f"{df_cleaned['Sulfate'].isna().sum()} (0.00%)",
        f"{df_cleaned['ph'].isna().sum()} (0.00%)",
        f"{df_cleaned['Trihalomethanes'].isna().sum()} (0.00%)",
        f"{df_cleaned['Solids'].max():.1f} ppm",
        f"{df_cleaned['Potability'].sum()} ({df_cleaned['Potability'].mean()*100:.2f}%)",
        "Zero Missing / Consistent Dtypes / Domain-Aligned"
    ]
})

summary_comparison

In [ ]:
# Comparative visual check on aggregate missing values
fig, ax = plt.subplots(figsize=(6, 4))
categories = ['Raw Dataset', 'Cleaned Dataset']
missing_totals = [df_raw.isna().sum().sum(), df_cleaned.isna().sum().sum()]

bars = ax.bar(categories, missing_totals, color=['#e74c3c', '#2ecc71'], width=0.5)
plt.title('Total Missing Values: Before vs. After Cleaning', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Count of Missing Values')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 20, f"{int(yval):,}", ha='center', va='bottom', fontweight='bold')
plt.ylim(0, max(missing_totals) + 200)
plt.tight_layout()
plt.show()

---
## 9. Oral Recitation and Defense Guide

### Question 1: Why did you use class-conditional median imputation instead of simply dropping missing rows?
> **Answer:** Over 30% of the dataset contained at least one missing chemical metric (`Sulfate` ~23.8%, `ph` ~15.0%). Applying listwise deletion (`dropna()`) would eliminate over 1,200 observations, significantly reducing statistical power and introducing severe selection bias (since missingness reflects selective testing rather than random failure). Grouping by `Potability` preserves the characteristic chemical signatures of potable vs. contaminated water without cross-class leakage.

### Question 2: Why is the median preferred over the mean for aquatic chemical metrics?
> **Answer:** Environmental chemical concentrations are heavily skewed by regional geological mineral deposits and industrial discharge. The mean is sensitive to extreme values, whereas the median provides a robust, outlier-resistant measure of central tendency.

### Question 3: How does domain knowledge justify setting pH boundaries strictly between 0 and 14?
> **Answer:** pH is fundamentally defined as the negative logarithm of hydrogen ion activity ($-\log_{10}[a_{H^+}]$). In natural aqueous systems, valid readings strictly reside between 0 and 14 (potable drinking water is regulated by the WHO between 6.5 and 8.5). Values outside this range represent sensor hardware defects or calibration errors.

### Question 4: Why didn't you remove high Total Dissolved Solids using standard $1.5 \times \text{IQR}$?
> **Answer:** Natural groundwater aquifers and brackish estuaries often have legitimately elevated mineral concentrations (10,000–30,000 ppm TDS) without being measurement errors. A strict $1.5 \times \text{IQR}$ filter would erroneously eliminate valid ecological records. Using a conservative $3.0 \times \text{IQR}$ threshold retains natural geochemical variability while discarding true telemetry spikes (> 50,000 ppm).